# TSA_ch12_long_memory

**Published in:** Time Series Analysis  
**Author:** Daniel Traian Pele  
**Description:** Long memory processes and fractional integration — spectral methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import periodogram

COLORS = {'blue':'#1A3A6E','red':'#DC3545','green':'#2E7D32','amber':'#B5853F','orange':'#E6802E','purple':'#8E44AD','gray':'#666666'}

In [ ]:
def simulate_arfima(d, N, seed=0):
    """Simulate ARFIMA(0,d,0) via truncated MA representation."""
    rng = np.random.default_rng(seed)
    eps = rng.normal(0, 1, N + 500)
    max_lag = min(N, 500)
    # MA coefficients: psi_j = prod_{k=0}^{j-1} (k+d)/(k+1)
    psi = np.zeros(max_lag)
    psi[0] = 1.0
    for j in range(1, max_lag):
        psi[j] = psi[j-1] * (d + j - 1) / j
    x = np.convolve(eps, psi)[:N + 500]
    return x[500:]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
d_vals = [0.0, 0.3, 0.45]
cols   = [COLORS['gray'], COLORS['blue'], COLORS['red']]
N = 1024

for d, col in zip(d_vals, cols):
    x = simulate_arfima(d, N)
    fr, psd = periodogram(x)
    # Log-log periodogram (skip DC)
    fr_pos = fr[1:N//4]
    psd_pos = psd[1:N//4]
    axes[1].loglog(fr_pos, psd_pos, color=col, lw=1, alpha=0.8, label=f'd={d}')
    # Theoretical slope line
    f0 = fr_pos[5]
    p0 = fr_pos[5]**(-2*d) if d > 0 else np.ones(len(fr_pos))[5]
    axes[1].loglog(fr_pos, fr_pos**(-2*d) * psd_pos[5] / max(fr_pos[5]**(-2*d), 1e-12),
                   color=col, lw=1.5, linestyle='--', alpha=0.5)
    # ACF
    acf = np.array([np.corrcoef(x[:-h], x[h:])[0,1] if h > 0 else 1.0 for h in range(50)])
    axes[0].plot(acf, color=col, lw=1.8, label=f'd={d}')

axes[0].set_title('ACF of ARFIMA(0,d,0)')
axes[0].set_xlabel('Lag h')
axes[0].set_ylabel('Autocorrelation')
axes[1].set_title('Log-Log Periodogram (slope = -2d)')
axes[1].set_xlabel('log Frequency')
axes[1].set_ylabel('log PSD')

for ax in axes:
    ax.set_facecolor('none')
    ax.spines[['top','right']].set_visible(False)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.savefig('long_memory.pdf', bbox_inches='tight')
plt.show()